# FoodIntel Training Notebook

This notebook is organized by the project phases. Phase 1 currently implements environment setup, dataset download/fallback handling, and dataset merge only.

**To prevent Colab disconnecting:** Open your browser console (F12 -> Console) and paste this:

```javascript
setInterval(() => document.querySelector("colab-toolbar-button#connect").click(), 60000)
```

Run it once and keep the tab active. For long training runs, consider Colab Pro for priority GPU access and longer sessions.

**Saving rule:** every model checkpoint or processed dataset produced in later sections must also be copied to `/content/drive/MyDrive/FoodIntel/` because Colab runtime storage is temporary.

# Section 0: Environment Setup

In [ ]:
# -- 0.1 Verify GPU -----------------------------------------------------------
import torch
assert torch.cuda.is_available(), (
    "No GPU detected! Go to Runtime -> Change runtime type -> T4 GPU -> Save"
)
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"   Memory: {round(torch.cuda.get_device_properties(0).total_memory/1e9,1)} GB")

# -- 0.2 Mount Google Drive --------------------------------------------------
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/FoodIntel'
import os
for folder in ['models', 'data/raw', 'data/processed', 'data/feedback', 'logs']:
    os.makedirs(f'{DRIVE_ROOT}/{folder}', exist_ok=True)
print('Drive mounted and folders ready')

# -- 0.3 Clone / Pull Repo ---------------------------------------------------
import os
if not os.path.exists('/content/FoodIntel-Backend'):
    !git clone https://github.com/BiodunDevv/FoodIntel-Backend.git /content/FoodIntel-Backend
else:
    !cd /content/FoodIntel-Backend && git pull
%cd /content/FoodIntel-Backend

# -- 0.4 Install Dependencies -----------------------------------------------
!pip install -q -r requirements.txt
!pip install -q kaggle loguru imagehash tensorboard torchmetrics scikit-learn matplotlib seaborn Pillow tqdm

print('Environment ready')

# Section 1: Dataset Download & Merge

This section downloads what can be downloaded automatically, supports manual fallbacks for gated/unavailable datasets, and then calls `ml/data/download_datasets.py` to build `ml/data/raw/unified/` plus `ml/data/raw/dataset_manifest.json`.

Important notes:
- Keep Kaggle credentials out of git. Store `kaggle.json` at `DRIVE_ROOT/kaggle.json` or provide Colab environment secrets.
- Mendeley may require manual download because a stable direct file ID is not exposed here.
- FoodNET's public repo currently documents that the image dataset link is broken/available by request, so this notebook accepts a manual ZIP/folder fallback.

In [ ]:
# -- 1.1 Raw dataset folders -------------------------------------------------
import os
from pathlib import Path

RAW_ROOT = Path('/content/data/raw')
RAW_ROOT.mkdir(parents=True, exist_ok=True)
for folder in ['mendeley', 'kaggle_nigeria', 'foodnet', 'food101', 'feedback']:
    (RAW_ROOT / folder).mkdir(parents=True, exist_ok=True)

print(f'Raw data root: {RAW_ROOT}')

In [ ]:
# -- 1.2 Kaggle credentials --------------------------------------------------
from pathlib import Path
import json
import os

kaggle_dir = Path.home() / '.kaggle'
kaggle_dir.mkdir(parents=True, exist_ok=True)
drive_kaggle = Path(DRIVE_ROOT) / 'kaggle.json'
target_kaggle = kaggle_dir / 'kaggle.json'

if drive_kaggle.exists():
    !cp '{drive_kaggle}' '{target_kaggle}'
    !chmod 600 '{target_kaggle}'
    print('Kaggle credentials loaded from Drive.')
elif os.getenv('KAGGLE_USERNAME') and os.getenv('KAGGLE_KEY'):
    target_kaggle.write_text(json.dumps({
        'username': os.environ['KAGGLE_USERNAME'],
        'key': os.environ['KAGGLE_KEY'],
    }))
    !chmod 600 '{target_kaggle}'
    print('Kaggle credentials loaded from environment variables.')
else:
    print('No Kaggle credentials found. Kaggle downloads will be skipped unless kaggle.json is added to Drive.')

In [ ]:
# -- 1.3 Download / stage datasets ------------------------------------------
from pathlib import Path
import os

def has_kaggle_credentials() -> bool:
    return (Path.home() / '.kaggle' / 'kaggle.json').exists()

# Mendeley African Foods Dataset: use manual Drive ZIP fallback by default.
mendeley_zip = Path(DRIVE_ROOT) / 'data/raw/mendeley_african_foods.zip'
if mendeley_zip.exists():
    !cp '{mendeley_zip}' /content/mendeley_african_foods.zip
    !unzip -oq /content/mendeley_african_foods.zip -d /content/data/raw/mendeley/
    print('Mendeley dataset extracted from Drive ZIP.')
else:
    print('Mendeley ZIP not found in Drive. Manual fallback: upload to DRIVE_ROOT/data/raw/mendeley_african_foods.zip')

# Kaggle Nigeria Food AI Dataset.
if has_kaggle_credentials():
    !kaggle datasets download -d elinteerie/nigeria-food-ai-dataset -p /content/data/raw/ --force
    !unzip -oq /content/data/raw/nigeria-food-ai-dataset.zip -d /content/data/raw/kaggle_nigeria/
    print('Kaggle Nigeria Food AI dataset downloaded and extracted.')
else:
    print('Skipping Kaggle Nigeria Food AI download: no Kaggle credentials.')

# FoodNET metadata clone. Public repo does not currently include the image bundle.
if not Path('/content/data/raw/foodnet/.git').exists():
    !git clone --depth 1 https://github.com/regchukwuka/FoodNET.git /content/data/raw/foodnet || true
else:
    !cd /content/data/raw/foodnet && git pull || true
foodnet_zip = Path(DRIVE_ROOT) / 'data/raw/foodnet.zip'
if foodnet_zip.exists():
    !unzip -oq '{foodnet_zip}' -d /content/data/raw/foodnet/
    print('FoodNET manual ZIP extracted from Drive.')
else:
    print('FoodNET image ZIP not found in Drive. The public repo may only provide metadata/notebook files.')

# Food-101 subset through Kaggle food41.
if has_kaggle_credentials():
    !kaggle datasets download -d kmader/food41 -p /content/data/raw/ --force
    !unzip -oq /content/data/raw/food41.zip -d /content/data/raw/food101/ '*pizza/*' '*sushi/*' '*fried_rice/*' '*hamburger/*' '*soup/*'
    print('Food-101 subset downloaded and extracted.')
else:
    print('Skipping Food-101 Kaggle download: no Kaggle credentials.')

# Optional app feedback data copied from Drive for future retraining-aware merges.
feedback_drive = Path(DRIVE_ROOT) / 'data/feedback'
if feedback_drive.exists():
    !cp -R '{feedback_drive}/.' /content/data/raw/feedback/ || true
    print('Feedback folder staged from Drive.')

In [ ]:
# -- 1.4 Merge datasets into unified ImageFolder -----------------------------
!python ml/data/download_datasets.py \
    --raw-root /content/data/raw \
    --output-dir ml/data/raw/unified \
    --manifest-path ml/data/raw/dataset_manifest.json \
    --class-map ml/nigerian_class_map.extensive.json \
    --clear-output

# Persist Phase 1 outputs to Google Drive.
!mkdir -p '{DRIVE_ROOT}/data/raw'
!cp -R ml/data/raw/unified '{DRIVE_ROOT}/data/raw/'
!cp ml/data/raw/dataset_manifest.json '{DRIVE_ROOT}/data/raw/dataset_manifest.json'

import os
assert os.path.exists('ml/data/raw/dataset_manifest.json'), 'Dataset manifest was not created.'
assert os.path.exists(f'{DRIVE_ROOT}/data/raw/dataset_manifest.json'), 'Manifest was not copied to Drive.'
print('Phase 1 complete. Share the summary table above before moving to Phase 2.')

# Section 2: Preprocessing & Augmentation

Placeholder for Phase 2. Do not run until the Phase 1 class summary has been reviewed.

# Section 3: Model Training

Placeholder for Phase 3.

# Section 4: Evaluation & Confusion Matrix

Placeholder for Phase 4.

# Section 5: Retraining on Feedback Data

Placeholder for Phase 5.

# Section 6: Export & Deploy to Backend

Placeholder for Phase 6.